 - AutoEncoder Test

# Libs

In [ ]:
import sys
sys.path.append("../libs/")
sys.path.append("../")

%env TF_ENABLE_ONEDNN_OPTS=0

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kurtosis, skew, entropy, linregress
from collections import defaultdict

import pywt

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans, DBSCAN, HDBSCAN, SpectralClustering, AgglomerativeClustering, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE, MDS, Isomap
from sklearn.decomposition import PCA, KernelPCA
from sklearn.impute import KNNImputer
from sklearn.metrics import euclidean_distances

from umap import UMAP

from tsfresh.feature_extraction import extract_features, EfficientFCParameters
from tsfresh.utilities.dataframe_functions import impute

import futurai_ppd as ppd
import futurai_utils as utils
from futurai_ml_dev import FuturaiML

DIR_DATA = os.getcwd() + "/data/"

## Functions

In [ ]:
def localizar_periodos_com_anomalia(futurai, result):
    df_pred = pd.DataFrame({"timestamp": result["timestamp"], "phi": result["phi"]})
    df_pred["timestamp"] = pd.to_datetime(df_pred["timestamp"], format="%Y-%m-%d %H:%M:%S")
    df_pred = df_pred.set_index('timestamp')
    df_pred = df_pred.resample("1T").asfreq()
    df_pred = df_pred.fillna(0)
    df_pred.reset_index(inplace=True)
    data_max = df_pred["timestamp"].max()
    data_min = df_pred["timestamp"].min()
    data_aux = data_min

    ##### onde se localiza a anomalia e preenche lista com datas de começo(onde phi>lim) e fim (onde phi<lim) #####
    list_datas: list[list[datetime, datetime]] = list()
    while data_aux <= data_max:
        mask = (df_pred["timestamp"] >= data_aux) & (
            df_pred["phi"] > futurai.phi_lim
        )
        df_aux2 = df_pred.loc[mask]
        if not df_aux2.empty:
            data_comeco = df_aux2["timestamp"].min()
        else:
            break

        mask = (df_pred["timestamp"] >= data_comeco) & (
            df_pred["phi"] <= futurai.phi_lim
        )
        df_aux3 = df_pred.loc[mask]
        
        if not df_aux3.empty:
            data_depois = df_aux3["timestamp"].min()
            data_aux = data_depois
        else:
            data_depois = df_aux2["timestamp"].max()
            list_datas.append([data_comeco, data_depois])
            break
            
        list_datas.append([data_comeco, data_depois])
    return list_datas

#########################################################################################################################################################
def unificar_anomalias(list_datas, anomaly_interval):
    list_datas_copy = list_datas.copy()
    dir = list_datas_copy[0][0]
    dfa = list_datas_copy[0][1]
    list_datas_copy.pop(0)

    list_datas_final = []

    anomalias_unificadas = [[(dir, dfa)]]
    idx_anomalias_unificadas = 0
    for per in list_datas_copy:
        di = per[0]
        df = per[1]
        interval = (di - dfa).total_seconds() / 60
        if interval < 0:
            pass
        if interval < anomaly_interval:
            dfa = df

            anomalias_unificadas[idx_anomalias_unificadas].append((di, df))
        else:
            list_datas_final.append([dir, dfa])
            dir = di
            dfa = df

            anomalias_unificadas.append(list())
            idx_anomalias_unificadas += 1
            anomalias_unificadas[idx_anomalias_unificadas].append((di, df))
    list_datas_final.append([dir, dfa])
    list_datas_final.reverse()
    
    dict_anomalias_unificadas = dict()
    date_fmt = '%Y-%m-%d %H:%M:%S'
    for anomalia_unificada in anomalias_unificadas:
        num_anomalias = len(anomalia_unificada)
        if num_anomalias <= 1:
            continue
        di = anomalia_unificada[0][0]
        df = anomalia_unificada[-1][1]
        di_str = di.strftime(date_fmt)
        df_str = df.strftime(date_fmt)
        key = f"{di_str} - {df_str}"
        dict_anomalias_unificadas.update({key: anomalia_unificada})

    return list_datas_final, dict_anomalias_unificadas

#########################################################################################################################################################
def periods_above_threshold(df_data, variable_datetime, predictions, model):
    """
    Identify periods in the DataFrame where values exceed the threshold.

    Args:
        df_data (pd.DataFrame): The input DataFrame with variables values.
        predictions (dict): The result dictionary containing predictions results and timestamps.
        model (object): The model object containing the threshold value.

    Returns:
        pd.DataFrame: A DataFrame containing only the periods above the threshold.
    """
    phi_values = np.array(predictions['phi'])
    indexes = np.where(phi_values > model.phi_lim)[0]
    indexes_expand = []
    window=1
    for idx in indexes:
        for i in range(idx - window, idx + window + 1):
            if 0 <= i < len(phi_values):
                indexes_expand.append(i)
    indexes_expand = sorted(set(indexes_expand))
    timestamps_above_threshold = np.array(predictions['timestamp'])[indexes_expand]

    df_data[variable_datetime] = pd.to_datetime(df_data[variable_datetime])
    timestamps_above_threshold = pd.to_datetime(timestamps_above_threshold)

    df_filtered = df_data[df_data[variable_datetime].isin(timestamps_above_threshold)]

    return df_filtered

#########################################################################################################################################################
def extract_tsfresh_features_robust(df_anomaly, anomaly_id=0):
    """
    Extrai features robustas para diagnóstico industrial usando tsfresh.
    
    Args:
        df_anomaly (pd.DataFrame): DataFrame com índice temporal e colunas de sensores.
        anomaly_id (int/str): Identificador da anomalia (útil se for processar em lote depois).
        
    Returns:
        dict: Dicionário com as features extraídas e tratadas (sem NaNs).
    """
    
    # 1. Validação Básica
    if df_anomaly.empty:
        return {}

    # 2. Preparação dos Dados (Wide -> Long Format)
    # O tsfresh exige formato: [id, time, kind (sensor), value]
    df_process = df_anomaly.copy()
    
    # Se o índice não tiver nome, damos um nome padrão para poder resetar
    if df_process.index.name is None:
        df_process.index.name = 'timestamp'
        
    df_long = (
        df_process
        .reset_index()
        .melt(id_vars=df_process.index.name, var_name='kind', value_name='value')
    )
    
    # Adiciona ID único para este trecho de dados
    df_long['id'] = anomaly_id
    
    # Garante que o tempo está no formato correto para ordenação
    time_col = df_process.index.name
    
    # 3. Definição do Dicionário de Features (Industrial Settings)
    # Ajustei a sintaxe para garantir compatibilidade total
    industrial_fc_parameters = {
        # --- Estatísticas Básicas (Amplitude) ---
        "mean": None,
        "median": None,
        "standard_deviation": None, # Variância removida (redundante)
        "minimum": None,
        "maximum": None,
        "root_mean_square": None, # Ouro para vibração
        
        # --- Dinâmica e Tendência (Degradação) ---
        "mean_abs_change": None, # Volatilidade
        # agg_linear_trend é mais robusto que linear_trend simples para ruído
        "agg_linear_trend": [{"attr": "slope", "chunk_len": 50, "f_agg": "mean"}],
        
        # --- Forma da Distribuição (Detecção de Outliers/Impactos) ---
        "skewness": None, # Assimetria (ótimo para falhas unilaterais)
        "kurtosis": None, # Achatamento (ótimo para impactos/batidas)
        "quantile": [{"q": 0.05}, {"q": 0.95}], # Foco nas caudas extremas
        
        # --- Comportamento Temporal & Complexidade (O que faltava) ---
        "binned_entropy": [{"max_bins": 10}], # Mede o "caos" ou saúde do sistema
        "longest_strike_above_mean": None,
        "number_peaks": [{"n": 3}, {"n": 5}], # Contagem de picos locais
        
        # --- Frequência Avançada (Densidade Espectral) ---
        # Substitui coeficientes FFT crus por PSD (Power Spectral Density)
        # Captura energia em diferentes bandas de frequência
        "spkt_welch_density": [{"coeff": 2}, {"coeff": 5}, {"coeff": 8}], 
        
        # Zero crossing rate (importante para oscilação mecânica)
        "count_above_mean": None, 
    }

    # 4. Extração
    try:
        # n_jobs=0 evita overhead de multiprocessamento para dataframes pequenos
        X = extract_features(
            df_long,
            column_id="id",
            column_sort=time_col,
            column_kind="kind",
            column_value="value",
            default_fc_parameters=industrial_fc_parameters,
            disable_progressbar=True,
            n_jobs=0 
        )
        
        # 5. Imputação (CRUCIAL)
        # Substitui NaNs (gerados por divisões por zero ou séries constantes) pela mediana
        X = impute(X)
        
        # Retorna como dicionário plano
        return X.iloc[0].to_dict()

    except Exception as e:
        print(f"Erro na extração de features para ID {anomaly_id}: {e}")
        return {}

#########################################################################################################################################################
def extract_features_manually(df_anomaly):
    features = {}
    for col in df_anomaly.columns:
        # Standard Features
        features[f"{col}_mean"] = df_anomaly[col].mean()
        features[f"{col}_std"] = df_anomaly[col].std()
        features[f"{col}_min"] = df_anomaly[col].min()
        features[f"{col}_max"] = df_anomaly[col].max()
        features[f"{col}_range"] = df_anomaly[col].max() - df_anomaly[col].min()
        features[f"{col}_cv"] = df_anomaly[col].std() / df_anomaly[col].mean() if df_anomaly[col].mean() != 0 else np.nan
        
        # Estatísticas de forma
        features[f"{col}_skew"] = skew(df_anomaly[col], bias=False)
        features[f"{col}_kurtosis"] = kurtosis(df_anomaly[col], bias=False)
        
        # Energia (soma dos quadrados)
        features[f'{col}_energy'] = np.sum(np.square(df_anomaly[col]))
        # RMS (Root Mean Square)
        features[f'{col}_rms'] = np.sqrt(np.mean(np.square(df_anomaly[col])))
        # Entropia de Shannon
        hist, bin_edges = np.histogram(df_anomaly[col], bins='auto', density=True)
        hist = hist[hist > 0]  # remove zeros
        features[f'{col}_entropy'] = entropy(hist)
    return features


#########################################################################################################################################################
def visualize_tsne(
    X_scaled,
    labels=None,
    perplexity=30.0,
    learning_rate='auto',
    max_iter=1000,
    random_state=42,
    method='barnes_hut',
    angle=0.5,
    metric='euclidean',
    verbose=1
):
    """
    Visualiza dados de alta dimensão usando t-SNE (scikit-learn >= 1.4 compatível).
    
    Parâmetros:
    -----------
    X_scaled : array-like, shape (n_samples, n_features)
        Dados normalizados.
    labels : array-like, opcional
        Rótulos (clusters ou classes) para colorir os pontos.
    perplexity : float
        Controla o equilíbrio entre vizinhanças locais e globais (típico: 5–50).
    learning_rate : float ou 'auto'
        Taxa de aprendizado (200–1000 normalmente).
    max_iter : int
        Número máximo de iterações de otimização.
    random_state : int
        Semente para reprodutibilidade.
    method : str
        'barnes_hut' (rápido, padrão) ou 'exact' (mais preciso).
    angle : float
        Parâmetro de trade-off velocidade/precisão no método 'barnes_hut' (0.2–0.8).
    metric : str
        Métrica de distância (ex: 'euclidean', 'cosine', 'manhattan').
    verbose : int
        0 = silencioso | 1 = imprime progresso.

    Retorna:
    --------
    np.ndarray
        Coordenadas 2D projetadas pelo t-SNE.
    """
    if verbose:
        print(f"Executando t-SNE (perplexity={perplexity}, learning_rate={learning_rate}, max_iter={max_iter})...")

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        early_exaggeration=12.0,
        learning_rate=learning_rate,
        max_iter=max_iter,
        n_iter_without_progress=500,
        min_grad_norm=1e-7,
        metric=metric,
        init='pca',
        random_state=random_state,
        method=method,
        angle=angle,
        verbose=verbose,
        n_jobs=None
    )

    X_embedded = tsne.fit_transform(X_scaled)

    if verbose:
        print("t-SNE concluído!")

    # --- Plot 2D ---
    plt.figure(figsize=(8, 6))
    if labels is not None:
        n_labels = len(np.unique(labels))
        palette = sns.color_palette("husl", n_labels if n_labels > 1 else 2)
        sns.scatterplot(
            x=X_embedded[:, 0],
            y=X_embedded[:, 1],
            hue=labels,
            palette=palette,
            s=40,
            alpha=0.85,
            edgecolor='none'
        )
        plt.legend(title="Cluster / Classe", bbox_to_anchor=(1.05, 1), loc='upper left')
    else:
        plt.scatter(X_embedded[:, 0], X_embedded[:, 1], s=40, alpha=0.7, c='red')

    plt.title("Visualização com t-SNE")
    plt.xlabel("Componente 1 (t-SNE)")
    plt.ylabel("Componente 2 (t-SNE)")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    return X_embedded

#########################################################################################################################################################

def visualize_umap(
    X_scaled,
    labels=None,
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric='euclidean',
    random_state=42,
    verbose=True
):
    """
    Visualiza dados de alta dimensão usando UMAP (Uniform Manifold Approximation and Projection).
    
    Geralmente é mais rápido que o t-SNE e preserva melhor a estrutura global dos dados.
    
    Parâmetros:
    -----------
    X_scaled : array-like, shape (n_samples, n_features)
        Dados normalizados (Features extraídas ou PCA).
    labels : array-like, opcional
        Rótulos (clusters ou classes) para colorir os pontos.
    n_neighbors : int
        Equivalente ao 'perplexity' do t-SNE. Controla o tamanho da vizinhança local.
        - Valores baixos (2-10): Foca em estrutura local (detalhes finos).
        - Valores altos (30-100): Foca em estrutura global (visão geral).
    min_dist : float
        Distância mínima entre pontos no espaço projetado (0.0 a 0.99).
        - Baixo (ex: 0.1): Clusters mais compactos e aglomerados.
        - Alto (ex: 0.5): Pontos mais espalhados, preserva melhor a topologia.
    n_components : int
        Dimensão final (padrão 2 para visualização).
    metric : str
        Métrica de distância (ex: 'euclidean', 'manhattan', 'cosine').
    random_state : int
        Semente para reprodutibilidade.
    verbose : bool
        Se True, imprime o progresso.

    Retorna:
    --------
    np.ndarray
        Coordenadas 2D projetadas pelo UMAP.
    """
    
    if verbose:
        print(f"Executando UMAP (n_neighbors={n_neighbors}, min_dist={min_dist}, metric={metric})...")

    # Instancia o redutor UMAP
    reducer = UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric=metric,
        random_state=random_state,
        verbose=verbose
    )

    # Ajusta e transforma os dados
    X_embedded = reducer.fit_transform(X_scaled)

    if verbose:
        print("UMAP concluído!")

    # --- Plot 2D (Mantendo o estilo do seu código original) ---
    plt.figure(figsize=(8, 6))
    
    if labels is not None:
        n_labels = len(np.unique(labels))
        # Paleta segura para evitar erro se houver apenas 1 cluster
        palette = sns.color_palette("husl", n_labels if n_labels > 1 else 2)
        
        sns.scatterplot(
            x=X_embedded[:, 0],
            y=X_embedded[:, 1],
            hue=labels,
            palette=palette,
            s=40,
            alpha=0.85,
            edgecolor='none'
        )
        # Posiciona a legenda fora do gráfico para não cobrir dados
        plt.legend(title="Cluster / Classe", bbox_to_anchor=(1.05, 1), loc='upper left')
    else:
        plt.scatter(X_embedded[:, 0], X_embedded[:, 1], s=40, alpha=0.7, c='blue')

    plt.title(f"Visualização com UMAP (neighbors={n_neighbors}, min_dist={min_dist})")
    plt.xlabel("Componente 1 (UMAP)")
    plt.ylabel("Componente 2 (UMAP)")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    return X_embedded

#########################################################################################################################################################

def plot_variable_by_cluster(df_data, labels, variable_name, timestamp_col='timestamp'):
    """
    Plota o comportamento de uma variável específica para cada cluster.
    
    Args:
        df_data: DataFrame 'df_dataset_above_threshold' (com coluna 'anomaly_id').
        labels: Lista com os labels dos clusters (na ordem dos anomaly_ids).
        variable_name: Nome da coluna/variável que você quer analisar.
        timestamp_col: Nome da coluna de tempo (para eixo X).
    """
    
    cluster_map = {i: label for i, label in enumerate(labels)}
    
    df_plot = df_data.copy()
    df_plot['cluster'] = df_plot['anomaly_id'].map(cluster_map)
    
    unique_clusters = sorted(list(set(labels)))
    n_clusters = len(unique_clusters)
    
    fig, axes = plt.subplots(n_clusters, 1, figsize=(12, 4 * n_clusters), sharex=False)
    if n_clusters == 1: axes = [axes]
    
    for ax, cluster_id in zip(axes, unique_clusters):
        
        df_cluster = df_plot[df_plot['cluster'] == cluster_id]
        anom_ids = df_cluster['anomaly_id'].unique()
        
        for a_id in anom_ids:
            subset = df_cluster[df_cluster['anomaly_id'] == a_id]
            
            x_axis = np.arange(len(subset))
            
            ax.plot(x_axis, subset[variable_name].values, alpha=0.3, color='blue', linewidth=1)
            
        # Adiciona a "Média" do cluster (linha mais grossa) para referência visual
        # Nota: Isso exige reamostragem se os tamanhos forem diferentes, aqui é apenas visual
        
        ax.set_title(f"Cluster {cluster_id} - Variável: {variable_name} ({len(anom_ids)} eventos)")
        ax.set_xlabel("Tempo Relativo (amostras)")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Load Data

In [ ]:
process_name = "Formação"
timestamp="TIMESTAMP"

list_colomuns_drop = ['l24_cl2_flowmetergm2', 'l24_cl2_mixrelation', 'l14_cl2_flowmetergm2','metal_dieffensor','espessura_receita_prensa']

df_dataset = ppd.load_dataset_principal(DIR_DATA+process_name+"_old.csv", list_colomuns_drop, timestamp, dropna=True, use_chunks=True, chunksize=10000)
df_dataset

# Set ON/OFF var

In [ ]:
pre_process = []
pp_var_ref_desligado = "l18-b01-04p"
pp_valor_ref_desligado = 40
pp_tempo_ref_desligado = 0
pp_pre_corte_transitorio = 30
pp_pos_corte_transitorio = 30
pre_process.append(  
{
   "after_cut": pp_pos_corte_transitorio,
   "interval_off": pp_tempo_ref_desligado,
   "limit_off": pp_valor_ref_desligado,
   "pre_cut": pp_pre_corte_transitorio,
   "variable_off": pp_var_ref_desligado
  })

# TAGs and descriptions

In [ ]:
df_sistema, df_sistema_drop =  ppd.set_tags_config(df_dataset,DIR_DATA+process_name+"_subsistema.csv")
df_sistema

# Training periods

## Period 1

In [ ]:
# train 1
start_date_train = pd.to_datetime('2025-07-13 00:00:00') 
end_date_train = pd.to_datetime('2025-07-14 00:00:00')

mask = (df_dataset[timestamp] >= start_date_train) & (df_dataset[timestamp] <= end_date_train)
df_train = df_dataset.loc[mask]

# pre - processamento
print(df_train.shape)
list_periods_train = []
for pro in pre_process:
    df_train,_,list_aux = ppd.drop_transitorio_desligado(df_train,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]
df_train.reset_index(inplace=True, drop=True)
print(df_train.shape)

eixoX_train = df_train[timestamp]
df_train = df_train.drop(timestamp,axis=1)

## Period 2

In [ ]:
#train 2
start_date_train2 = pd.to_datetime('2025-09-05 00:00:00') 
end_date_train2 = pd.to_datetime('2025-09-05 12:00:00')

mask = (df_dataset[timestamp] > start_date_train2) & (df_dataset[timestamp] <= end_date_train2)
df_train2 = df_dataset.loc[mask]

# pre - processamento
print(df_train2.shape)
list_periods_train = []
for pro in pre_process:
    df_train2,_,list_aux = ppd.drop_transitorio_desligado(df_train2,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]
df_train2.reset_index(inplace=True, drop=True)
print(df_train2.shape)

eixoX_train2 = df_train2[timestamp]
df_train2 = df_train2.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train2], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train2], ignore_index=True)

## Período 3

In [ ]:
start_date_train3 = pd.to_datetime('2025-08-15 00:00:00') 
end_date_train3 = pd.to_datetime('2025-08-15 06:00:00')

mask = (df_dataset[timestamp] > start_date_train3) & (df_dataset[timestamp] <= end_date_train3)
df_train3 = df_dataset.loc[mask]

# pre - processamento
print(df_train3.shape)
list_periods_train = []
for pro in pre_process:
    df_train3,_,list_aux = ppd.drop_transitorio_desligado(df_train3,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]
df_train3.reset_index(inplace=True, drop=True)
print(df_train3.shape)

eixoX_train3 = df_train3[timestamp]
df_train3 = df_train3.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train3], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train3], ignore_index=True)

## Período 4

In [ ]:
start_date_train4 = pd.to_datetime('2025-09-09 03:00:00') 
end_date_train4 = pd.to_datetime('2025-09-09 04:00:00')

mask = (df_dataset[timestamp] > start_date_train4) & (df_dataset[timestamp] <= end_date_train4)
df_train4 = df_dataset.loc[mask]

# pre - processamento
print(df_train4.shape)
list_periods_train = []
for pro in pre_process:
    df_train4,_,list_aux = ppd.drop_transitorio_desligado(df_train4,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]
df_train4.reset_index(inplace=True, drop=True)
print(df_train4.shape)

eixoX_train4 = df_train4[timestamp]
df_train4 = df_train4.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train4], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train4], ignore_index=True)

# Fit Model

In [ ]:
# Instacinamento da Classe
gain = 3.5
nc = 0
futurai = FuturaiML(nc,gain)

# Gerando o modelo
futurai.fit(df_train)

# Print dos limiares do modelo
print("Modelo")
print("T²: {:.2f}".format(futurai.t2_lim))
print("Q: {:.2f}".format(futurai.q_lim))
print("Phi: {:.2f}".format(futurai.phi_lim))
print("Componentes: {:}".format(futurai.nc))     

# Predict

In [ ]:
start_date = pd.to_datetime("2025-09-12 14:00:00")
end_date = pd.to_datetime("2025-12-30 19:21:00")

mask = (df_dataset[timestamp] >= start_date) & (df_dataset[timestamp] <= end_date)
df_test = df_dataset.loc[mask]

# pre - processamento
print(df_test.shape)
list_periods_test = []
for pro in pre_process:
    df_test,_,list_aux = ppd.drop_transitorio_desligado(df_test,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_test = [*list_periods_test,*list_aux]
print(df_test.shape)

eixoX_test = df_test[timestamp]
df_test_aux = df_test.copy()
df_test_aux.set_index(timestamp, inplace=True)
df_test = df_test.drop(timestamp,axis=1)
df_test.reset_index(inplace=True, drop=True)

result = futurai.predict(df_test,eixoX_test)

## Plot predictions

In [ ]:
list_periods_test = ppd.merge_periods(list_periods_test)
fig_all_period, _ = utils.dev_graph_predict(result["phi"], result["timestamp"], futurai.phi_lim, " ", start_date, end_date, list_periods=False, plot_anomalies=False)
fig_all_period.show()

# Locate Anomalies

In [ ]:
# list_datas = localizar_periodos_com_anomalia(futurai, result)
# anomaly_interval = 1440
# anomaly_periods, _ = unificar_anomalias(list_datas, anomaly_interval)
# print(f"Qtd. anomalias: {len(anomaly_periods)}")
# anomaly_periods.reverse()

In [ ]:
anomaly_periods = [
    [pd.Timestamp('2025-09-12 14:03:00'), pd.Timestamp('2025-09-24 00:07:00')],
    [pd.Timestamp('2025-10-03 12:04:00'), pd.Timestamp('2025-11-01 16:00:00')],
    [pd.Timestamp('2025-11-04 18:20:00'), pd.Timestamp('2025-11-26 10:00:00')],
    [pd.Timestamp('2025-12-07 20:35:00'), pd.Timestamp('2025-12-11 18:00:00')]
]

# Slide Window

In [ ]:
start_date = pd.to_datetime("2025-01-01 00:00:00")
end_date = pd.to_datetime("2025-12-30 00:00:00")

mask = (df_test_aux.index >= start_date) & (df_test_aux.index <= end_date)
df_subset = df_test_aux.loc[mask]

start_date = df_subset.index.min()
end_date = df_subset.index.max()
window_periods = []

current = start_date
while current < end_date:
    period_end = current + pd.Timedelta(hours=72)
    window_periods.append([current, period_end])
    current = period_end

window_periods

### Get periods above threshold and extract features

In [ ]:
list_dates = anomaly_periods ## anomaly_periods | window_periods

In [ ]:
rows = []
dfs_above_threshold_list = []
dfs_anomalies_list = []

for index, datas in enumerate(list_dates):
    ## preprocessing
    mask = (df_dataset[timestamp] >= datas[0]) & (df_dataset[timestamp] <= datas[1])
    df_anom = df_dataset.loc[mask]
    
    for pro in pre_process:
        df_anom,_,_ = ppd.drop_transitorio_desligado(df_anom,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    
    df_anom_with_timestamp = df_anom.copy()
    eixoX_anom = df_anom[timestamp]
    df_anom = df_anom.drop(timestamp,axis=1)
    df_anom.reset_index(inplace=True, drop=True)
    
    ## predict
    result = futurai.predict(df_anom,eixoX_anom)
    
    ## filtering periods above threshold
    df_filtered = periods_above_threshold(df_anom_with_timestamp, timestamp, result, futurai)
    
    if not df_filtered.empty:
        df_to_concat = df_filtered.copy()
        df_to_concat['anomaly_id'] = index
        dfs_above_threshold_list.append(df_to_concat)

    if not df_anom.empty:
        df_anom_to_concat = df_anom_with_timestamp.copy()
        df_anom_to_concat['anomaly_id'] = index
        dfs_anomalies_list.append(df_anom_to_concat)
    
    ## extract features
    df_filtered.set_index(timestamp, inplace=True)
    feats = extract_tsfresh_features_robust(df_filtered, anomaly_id=index)
    rows.append(feats)

if dfs_above_threshold_list and dfs_anomalies_list:
    df_dataset_above_threshold = pd.concat(dfs_above_threshold_list, ignore_index=True)
    df_dataset_anomalies = pd.concat(dfs_anomalies_list, ignore_index=True)
else:
    df_dataset_above_threshold = pd.DataFrame()
    df_dataset_anomalies = pd.DataFrame()

df_features = pd.DataFrame(rows)

print("Shape raw data above threshold:", df_dataset_above_threshold.shape) ## dados apenas nos períodos acima do limiar nas anomalias
print("Shape features:", df_features.shape) ## dados das features que foram extraídas
print("Shape raw data anomalies: ", df_dataset_anomalies.shape) ## dados de todo período das anomalias

#### PFA (Principal Feature Analysis)

In [ ]:
## PFA implementation of Time2Feat
class PFA(object):
    def __init__(self, q=None):
        self.q = q

    def fit(self, X, expl_var_value: float):
        if not self.q:
            self.q = X.shape[1]

        sc = StandardScaler()
        X_trans = sc.fit_transform(X)
        # Choice of the Explained Variance
        pca = PCA(expl_var_value)
        pca.fit(X_trans)
        princComp = len(pca.explained_variance_ratio_)
        A_q = np.abs(pca.components_.T)

        kmeans = KMeans(n_clusters=princComp, n_init=10)
        kmeans.fit(A_q)
        clusters = kmeans.predict(A_q)
        cluster_centers = kmeans.cluster_centers_

        dists = defaultdict(list)
        for i, c in enumerate(clusters):
            dist = euclidean_distances([A_q[i, :]], [cluster_centers[c, :]])[0][0]
            dists[c].append((i, dist))

        self.indices_ = [sorted(f, key=lambda x: x[1])[0][0] for f in dists.values()]
        self.features_ = X_trans[:, self.indices_]
        list_feat = []
        for x in self.indices_:
            list_feat.append(X.columns[x])

        return list_feat, pca.explained_variance_ratio_


def pfa_scoring(df: pd.DataFrame, expl_var_selection: float):
    pfa = PFA()
    feat_pfa, expl_variance_ration = pfa.fit(df, expl_var_selection)
    return feat_pfa, expl_variance_ration

In [ ]:
df_dataset_above_threshold_ids = df_dataset_above_threshold.copy()
df_dataset_above_threshold.drop(columns=[timestamp, "anomaly_id"], inplace=True, axis=1, errors='ignore')
top_variables, _ = pfa_scoring(df_dataset_above_threshold, 0.95)
print("PFA top variables: ")
top_variables

##### Selected all variables

In [ ]:
scaler = StandardScaler()
X_all_variables_scaled = scaler.fit_transform(df_dataset_above_threshold)

print(f"shape: {X_all_variables_scaled.shape}")

##### Selected all features

In [ ]:
scaler = StandardScaler()
X_all_features_scaled = scaler.fit_transform(df_features)

print(f"shape: {X_all_features_scaled.shape}")

# DWT

In [ ]:
def extract_wavelet_features_variable_length(raw_data_list, wavelet='db4', desired_level=3):
    """
    Extrai features Wavelet de séries temporais com tamanhos variados.
    Ajusta automaticamente o nível de decomposição para caber na menor série.
    """
    feature_matrix = []
    min_len = min([x.shape[0] for x in raw_data_list])
    wavelet_obj = pywt.Wavelet(wavelet)
    max_safe_level = pywt.dwt_max_level(min_len, wavelet_obj.dec_len)
    final_level = min(desired_level, max_safe_level)
    
    if final_level < desired_level:
        print(f"Aviso: Nível reduzido de {desired_level} para {final_level} devido à série mais curta ({min_len} pts).")
    else:
        print(f"Nível de decomposição definido: {final_level} (Seguro para min_len={min_len})")

    print(f"Extraindo features de {len(raw_data_list)} janelas multivariadas...")
    
    for i, window in enumerate(raw_data_list):
        # window shape: (timestamps, variaveis)
        window_feats = []
        n_sensors = window.shape[1]
        
        for sensor_idx in range(n_sensors):
            signal = window[:, sensor_idx]
            coeffs = pywt.wavedec(signal, wavelet, level=final_level)

            for c in coeffs:
                # 1. Energia Média (Power): Soma Quadrados / N
                # Normalizar por len(c) para comparar janelas de durações diferentes
                energy = np.sum(np.square(c)) / len(c)
                
                # 2. Média
                avg = np.mean(c)
                
                # 3. Desvio Padrão
                std = np.std(c)
                
                # 4. Entropia de Shannon (Complexidade)
                p = np.abs(c) / (np.sum(np.abs(c)) + 1e-9)
                entropy = -np.sum(p * np.log2(p + 1e-9))
                
                window_feats.extend([energy, avg, std, entropy])
        
        feature_matrix.append(window_feats)
        
    return np.array(feature_matrix)

raw_data_list = []
valid_periods_indices = [] 

print("\n1. Extraindo dados brutos do DataFrame...")
for idx, (start_dt, end_dt) in enumerate(anomaly_periods):
    try:
        df_slice = df_test_aux.loc[start_dt:end_dt] ### 
        
        # mínimo de pontos para DWT
        if len(df_slice) > 30: 
            raw_data_list.append(df_slice.values)
            valid_periods_indices.append(idx)
        else:
            print(f"   Aviso: Período {idx} ignorado (muito curto). Tamanho {len(df_slice)}")
            
    except KeyError:
        print(f"   Erro: Período {idx} fora do intervalo.")


print("\n2. Extraindo features Wavelet (Tamanho Variável)...")
X_wavelet_features = extract_wavelet_features_variable_length(
    raw_data_list, 
    wavelet='db4', 
    desired_level=4  # Tenta nível 4 reduz se necessário
)
print(f"   Matriz de Features final: {X_wavelet_features.shape}")

scaler = StandardScaler()
X_wavelet_features = scaler.fit_transform(X_wavelet_features)

# Clustering

## Feature based

### Visualize Data

In [ ]:
X_visualize = X_all_features_scaled  ## X_all_features_scaled | X_wavelet_features

#### t-SNE on high dimensional dataset

In [ ]:
X_tsne = visualize_tsne(X_visualize, labels=None, perplexity=1, learning_rate='auto', max_iter=10000, random_state=42, method='exact', angle=0.5, metric='euclidean', verbose=0)

#### MDS on high dimensional dataset

In [ ]:
mds = MDS(n_components=2, n_init=5, random_state=42)
X_mds = mds.fit_transform(X_visualize)
print(X_mds.shape)

plt.figure(figsize=(8, 6))
plt.scatter(X_mds[:, 0], X_mds[:, 1], s=40, alpha=0.7, c='red')
plt.title("Visualização com MDS")
plt.xlabel("Componente 1 (MDS)")
plt.ylabel("Componente 2 (MDS)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

#### UMAP on high dimensional dataset

In [ ]:
X_umap = visualize_umap(X_visualize, labels=None, n_neighbors=2, min_dist=0.1, n_components=2, metric='euclidean', random_state=42, verbose=False)

### PCA to dimensionality reduction

In [ ]:
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_visualize)
nc = X_pca.shape[1]
print(f"Número de componentes principais: {nc}")

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=40, alpha=0.7, c='red')
plt.title("Visualização com PCA")
plt.xlabel("Componente 1 (PCA)")
plt.ylabel("Componente 2 (PCA)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Apply AutoEncoder to compress data

In [ ]:
X_flat = X_all_features_scaled ##

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# shallow AE
class ShallowAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=10):
        super(ShallowAutoencoder, self).__init__()
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, input_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        latent = self.relu(self.encoder(x))
        reconstructed = self.decoder(latent)
        return reconstructed, latent

def best_hidden_dim(X_input, input_dim, candidate_dims):
    losses = []

    tensor_x = torch.Tensor(X_input)
    dataset = TensorDataset(tensor_x, tensor_x)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
    
    for h_dim in candidate_dims:
        model = ShallowAutoencoder(input_dim, hidden_dim=h_dim)
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()
        
        for epoch in range(500): 
            for batch, _ in dataloader:
                optimizer.zero_grad()
                outputs, _ = model(batch)
                loss = criterion(outputs, batch)
                loss.backward()
                optimizer.step()
        
        model.eval()
        with torch.no_grad():
            reconstructed, _ = model(tensor_x)
            final_loss = criterion(reconstructed, tensor_x).item()
            
        losses.append(final_loss)
        print(f"Dim: {h_dim} -> Loss: {final_loss:.6f}")
        
    # Plot
    plt.figure(figsize=(8, 5))
    plt.plot(candidate_dims, losses, 'bo-')
    plt.title("Reconstruction Error vs Hidden Dim")
    plt.xlabel("Hidden Dim (Neurons)")
    plt.ylabel("MSE Loss")
    plt.grid(True)
    plt.show()

candidates = [2, 4, 8, 12, 16, 24, 32, 64, 128]
candidates = [c for c in candidates if c < X_flat.shape[1]]

best_hidden_dim(X_flat, X_flat.shape[1], candidates)

In [ ]:
tensor_x = torch.Tensor(X_flat)
dataset = TensorDataset(tensor_x, tensor_x) # AE Entrada = Saída
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# config
input_dim = X_flat.shape[1]
hidden_dim = 24
model = ShallowAutoencoder(input_dim, hidden_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# train
epochs = 1000
loss_history = []

for epoch in range(epochs):
    epoch_loss = 0
    for batch_features, _ in dataloader:
        optimizer.zero_grad()
        outputs, _ = model(batch_features)
        loss = criterion(outputs, batch_features)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    loss_history.append(epoch_loss / len(dataloader))

plt.plot(loss_history)
plt.title("Loss")
plt.show()

# get latent space
model.eval()
with torch.no_grad():
    _, latent_vectors = model(tensor_x)
    
# latent_vectors new data for clustering
X_neural_features = latent_vectors.numpy()

### Define parameters to run algorithms

In [ ]:
X_cluster = X_pca # X_all_features_scaled | X_wavelet_features | X_pca | X_neural_features

### K-Means

In [ ]:
def find_optimal_k_kmeans(X_scaled, k_min=1, k_max=10, plot=True):
    """
    Busca automática do número ótimo de clusters usando várias métricas de validação.
    
    Calcula:
    - Silhouette
    - Calinski-Harabasz
    - Inertia (Elbow)

    Exibe todos os gráficos em um único subplot.

    Returns:
    --------
    dict:
        scores (dict com listas de scores),
        best_models (dict com modelos KMeans para cada métrica)
    """

    ks = list(range(k_min, k_max + 1))

    # Armazena métricas
    results = {
        "silhouette": [],
        "calinski": [],
        "inertia": []
    }

    # Para salvar o modelo final por métrica
    best_models = {}

    for k in ks:
        kmeans = KMeans(n_clusters=k, random_state=42)
        labels = kmeans.fit_predict(X_scaled)

        inertia = kmeans.inertia_
        results["inertia"].append(inertia)

        # Silhouette requer K >= 2
        if k > 1:
            results["silhouette"].append(silhouette_score(X_scaled, labels))
            results["calinski"].append(calinski_harabasz_score(X_scaled, labels))
        else:
            results["silhouette"].append(np.nan)
            results["calinski"].append(np.nan)

    # -----------------------------
    # Determinação de cada K ótimo
    # -----------------------------
    best_k = {}

    best_k["silhouette"] = ks[np.nanargmax(results["silhouette"])]
    best_k["calinski"]   = ks[np.argmax(results["calinski"])]

    # Para o método do cotovelo:
    diffs = np.diff(results["inertia"])
    second_diffs = np.diff(diffs)
    elbow_k = np.argmin(second_diffs) + k_min + 1
    best_k["elbow"] = elbow_k

    # Modelos finais
    for metric_name, k_value in best_k.items():
        best_models[metric_name] = KMeans(n_clusters=k_value, random_state=42).fit(X_scaled)

    # -----------------------------
    # Plot em subplots
    # -----------------------------
    if plot:

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        ax1, ax2, ax3, ax4 = axes.ravel()

        # --- Silhouette ---
        ax1.plot(ks, results["silhouette"], marker='o')
        ax1.set_title("Silhouette Score")
        ax1.set_xlabel("K")
        ax1.set_ylabel("Silhouette")
        ax1.grid()

        # --- Calinski-Harabasz ---
        ax2.plot(ks, results["calinski"], marker='o')
        ax2.set_title("Calinski-Harabasz")
        ax2.set_xlabel("K")
        ax2.set_ylabel("Score")
        ax2.grid()

        # --- Elbow (Inertia) ---
        ax3.plot(ks, results["inertia"], marker='o')
        ax3.set_title("Método do Cotovelo (Inertia)")
        ax3.set_xlabel("K")
        ax3.set_ylabel("Inertia")
        ax3.grid()

        plt.tight_layout()
        plt.show()

    # Retorno organizado
    return {
        "best_k": best_k,
        "scores": results,
        "best_models": best_models
    }

In [ ]:
result_kmeans = find_optimal_k_kmeans(X_cluster, k_min=1, k_max=3) #silhouette | elbow | calinski

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
labels_kmeans = kmeans.fit_predict(X_cluster)

In [ ]:
X_umap = visualize_umap(X_cluster, labels=labels_kmeans, n_neighbors=5, min_dist=0.1, n_components=2, metric='euclidean', random_state=42, verbose=False)

In [ ]:
X_tsne = visualize_tsne(X_cluster, labels=labels_kmeans, perplexity=1, learning_rate='auto', max_iter=10000, random_state=42, method='exact', angle=0.5, metric='euclidean', verbose=0)

In [ ]:
cluster_colors = {
    0: 'rgba(31, 119, 180, 1)',   # Azul Forte
    1: 'rgba(255, 127, 14, 1)',   # Laranja Vivo
    2: 'rgba(44, 160, 44, 1)',    # Verde Floresta
    3: 'rgba(214, 39, 40, 1)',    # Vermelho Tijolo
    4: 'rgba(148, 103, 189, 1)',  # Roxo
    5: 'rgba(140, 86, 75, 0.6)',    # Marrom
    6: 'rgba(227, 119, 194, 0.6)',  # Rosa
    7: 'rgba(127, 127, 127, 0.6)',  # Cinza
    8: 'rgba(188, 189, 34, 0.6)',   # Verde Oliva
    9: 'rgba(23, 190, 207, 0.6)'    # Azul Teal/Ciano
}

for (start, end), label in zip(anomaly_periods, labels_kmeans):
    color = cluster_colors.get(label, 'rgba(128, 128, 128, 0.3)')
    
    fig_all_period.add_vrect(
        x0=start, 
        x1=end,
        fillcolor=color,
        opacity=1,           # Opacidade já controlada na cor (rgba) ou use float aqui se usar hex
        layer="below",       # Coloca a faixa atrás das linhas do gráfico
        line_width=0,        # Remove borda da faixa
    )

fig_all_period.show()

#### Interpret results

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import shap

X = pd.DataFrame(X_all_features_scaled, columns=df_features.columns)
y = labels_kmeans

rf_model = RandomForestClassifier(n_estimators=500, random_state=42)
rf_model.fit(X, y)

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer(X)

# plt.figure(figsize=(10, 6))
# plt.title("Importância Global das Features")
# shap.summary_plot(shap_values.values, X, plot_type="bar", show=True)

classes = rf_model.classes_

for i, cluster_label in enumerate(classes):
    mask = (y == cluster_label)
    X_cluster_i = X[mask]

    shap_cluster_i = shap_values.values[mask, :, i]

    plt.figure(figsize=(10, 6))
    plt.title(f"SHAP Beeswarm - Cluster {cluster_label}")
    shap.summary_plot(shap_cluster_i, X_cluster_i, show=True)

### Spectral Clustering

In [ ]:
def find_optimal_k_spectral(X_scaled, k_min=2, k_max=10, plot=True):
    """
    Busca automática do K ótimo para Spectral Clustering.
    
    Calcula:
    - Silhouette
    - Calinski-Harabasz

    Args:
        k_min (int): Mínimo de clusters (>= 2)
        k_max (int): Máximo de clusters
    """

    ks = list(range(k_min, k_max + 1))

    # Armazena métricas
    results = {
        "silhouette": [],
        "calinski": [],
    }

    # Para salvar o modelo final por métrica
    best_models = {}

    for k in ks:
        # affinity='nearest_neighbors' é robusto para geometrias complexas
        spectral = SpectralClustering(n_clusters=k, affinity='nearest_neighbors', 
                                      assign_labels='discretize', random_state=42, n_jobs=-1)
        labels = spectral.fit_predict(X_scaled)

        # Métricas de validação
        results["silhouette"].append(silhouette_score(X_scaled, labels))
        results["calinski"].append(calinski_harabasz_score(X_scaled, labels))

    # -----------------------------
    # Determinação de cada K ótimo
    # -----------------------------
    best_k = {}

    best_k["silhouette"] = ks[np.argmax(results["silhouette"])]
    best_k["calinski"]   = ks[np.argmax(results["calinski"])]


    # Modelos finais
    for metric_name, k_value in best_k.items():
        best_models[metric_name] = SpectralClustering(n_clusters=k_value, affinity='nearest_neighbors', 
                                                      assign_labels='discretize', random_state=42, n_jobs=-1).fit(X_scaled)

    # -----------------------------
    # Plot em subplots
    # -----------------------------
    if plot:
        
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        ax1, ax2 = axes.ravel()

        # --- Silhouette ---
        ax1.plot(ks, results["silhouette"], marker='o', color='blue')
        ax1.set_title("Silhouette Score")
        ax1.set_xlabel("K")
        ax1.set_ylabel("Silhouette")
        ax1.grid()

        # --- Calinski-Harabasz ---
        ax2.plot(ks, results["calinski"], marker='o', color='green')
        ax2.set_title("Calinski-Harabasz")
        ax2.set_xlabel("K")
        ax2.set_ylabel("Score")
        ax2.grid()

        plt.tight_layout()
        plt.show()

    return {
        "best_k": best_k,
        "scores": results,
        "best_models": best_models
    }

In [ ]:
resultado_spectral = find_optimal_k_spectral(X_cluster, k_min=2, k_max=10)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.neighbors import kneighbors_graph
from scipy.sparse import csgraph
from numpy import linalg as LA

# 1. Construindo a matriz de adjacências
G = kneighbors_graph(X_cluster, n_neighbors=5, include_self=True)
A = 0.5 * (G + G.T)

# 2. Construindo a Laplaciana Normalizada
L = csgraph.laplacian(A, normed=True).todense()

# 3. Obtendo os autovalores
values, _ = LA.eigh(L)

# Define quantos autovalores mostrar. 
# Regra: Mostra os primeiros 30 ou o tamanho total do dataset (o que for menor).
# Raramente precisamos avaliar mais que 30 clusters.
n_samples = len(values)
limit = min(n_samples, 30) 

# Eixo X dinâmico baseado no limite
x_axis = range(1, limit + 1)

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(x_axis, values[:limit], marker='o', s=50)
plt.plot(x_axis, values[:limit], linestyle='--', alpha=0.5) # Linha ajuda a ver o 'cotovelo'

plt.title('Eigengap Heuristic')
plt.xlabel('Índice do autovalor (k)')
plt.ylabel('Autovalor')
plt.xticks(x_axis) # Força mostrar todos os inteiros no eixo X
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
spectral = SpectralClustering(n_clusters=2, affinity='nearest_neighbors', assign_labels='discretize', random_state=42, n_jobs=-1)
labels_spectral = spectral.fit_predict(X_cluster)

In [ ]:
X_umap = visualize_umap(X_cluster, labels=labels_spectral, n_neighbors=4, min_dist=0.1, n_components=2, metric='euclidean', random_state=42, verbose=False)

In [ ]:
X_tsne = visualize_tsne(X_cluster, labels=labels_spectral, perplexity=4, learning_rate='auto', max_iter=10000, random_state=42, method='exact', angle=0.5, metric='euclidean', verbose=0)

#### Plot variables by clusters

In [ ]:
plot_variable_by_cluster(
    df_data=df_dataset_anomalies, 
    labels=labels_spectral, 
    variable_name='torque_ds', # nome da variavel
    timestamp_col=timestamp # variável de timestamp
)

In [ ]:
plot_variable_by_cluster(
    df_data=df_dataset_anomalies, 
    labels=labels_spectral, 
    variable_name='L1_2316', # nome da variavel
    timestamp_col=timestamp # variável de timestamp
)

In [ ]:
cluster_colors = {
    0: 'rgba(31, 119, 180, 0.6)',   # Azul Forte
    1: 'rgba(255, 127, 14, 0.6)',   # Laranja Vivo
    2: 'rgba(44, 160, 44, 0.6)',    # Verde Floresta
    3: 'rgba(214, 39, 40, 0.6)',    # Vermelho Tijolo
    4: 'rgba(148, 103, 189, 0.6)',  # Roxo
    5: 'rgba(140, 86, 75, 0.6)',    # Marrom
    6: 'rgba(227, 119, 194, 0.6)',  # Rosa
    7: 'rgba(127, 127, 127, 0.6)',  # Cinza
    8: 'rgba(188, 189, 34, 0.6)',   # Verde Oliva
    9: 'rgba(23, 190, 207, 0.6)'    # Azul Teal/Ciano
}

for (start, end), label in zip(anomaly_periods, labels_spectral):
    color = cluster_colors.get(label, 'rgba(128, 128, 128, 0.3)')
    
    fig_all_period.add_vrect(
        x0=start, 
        x1=end,
        fillcolor=color,
        opacity=1,           # Opacidade já controlada na cor (rgba) ou use float aqui se usar hex
        layer="below",       # Coloca a faixa atrás das linhas do gráfico
        line_width=0,        # Remove borda da faixa
    )

fig_all_period.show()

#### Interpret results

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import shap

X = pd.DataFrame(X_all_features_scaled, columns=df_features.columns)
y = labels_spectral

rf_model = RandomForestClassifier(n_estimators=500, random_state=42)
rf_model.fit(X, y)

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer(X)

# plt.figure(figsize=(10, 6))
# plt.title("Importância Global das Features")
# shap.summary_plot(shap_values.values, X, plot_type="bar", show=True)

classes = rf_model.classes_

for i, cluster_label in enumerate(classes):
    mask = (y == cluster_label)
    X_cluster_i = X[mask]

    shap_cluster_i = shap_values.values[mask, :, i]

    plt.figure(figsize=(10, 6))
    plt.title(f"SHAP Beeswarm - Cluster {cluster_label}")
    shap.summary_plot(shap_cluster_i, X_cluster_i, show=True)

### Gaussian Mixture Model (GMM)

In [ ]:
def find_optimal_k_gmm(X_scaled, k_min=2, k_max=10, plot=True):
    """
    Busca automática do K (componentes) ótimo para Gaussian Mixture Models.
    
    Calcula:
    - Silhouette
    - Calinski-Harabasz
    - BIC (Bayesian Information Criterion) -> Substitui a Inertia

    Args:
        k_min (int): Mínimo de clusters (>= 2)
        k_max (int): Máximo de clusters
    """

    ks = list(range(k_min, k_max + 1))

    # Armazena métricas
    results = {
        "silhouette": [],
        "calinski": [],
        "bic": [] # Substitui Inertia
    }

    # Para salvar o modelo final por métrica
    best_models = {}

    for k in ks:
        # n_init=10 reinicia o algoritmo 10 vezes para evitar mínimos locais
        gmm = GaussianMixture(n_components=k, random_state=42, n_init=10)
        gmm.fit(X_scaled)
        labels = gmm.predict(X_scaled)

        # Métricas de validação (Baseadas nas hard labels)
        results["silhouette"].append(silhouette_score(X_scaled, labels))
        results["calinski"].append(calinski_harabasz_score(X_scaled, labels))

        # --- BIC (Critério de Informação Bayesiano) ---
        # Penaliza modelos complexos. Quanto MENOR, melhor.
        results["bic"].append(gmm.bic(X_scaled))

    # -----------------------------
    # Determinação de cada K ótimo
    # -----------------------------
    best_k = {}

    best_k["silhouette"] = ks[np.argmax(results["silhouette"])]
    best_k["calinski"]   = ks[np.argmax(results["calinski"])]
    
    # Para o BIC, o melhor valor é simplesmente o MÍNIMO (não precisa de derivada/cotovelo)
    best_k["bic"] = ks[np.argmin(results["bic"])]

    # Modelos finais
    for metric_name, k_value in best_k.items():
        best_models[metric_name] = GaussianMixture(n_components=k_value, random_state=42, n_init=10).fit(X_scaled)

    # -----------------------------
    # Plot em subplots
    # -----------------------------
    if plot:
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        ax1, ax2, ax3, ax4 = axes.ravel()

        # --- Silhouette ---
        ax1.plot(ks, results["silhouette"], marker='o', color='blue')
        ax1.set_title("Silhouette Score (Maior é melhor)")
        ax1.set_xlabel("K (Componentes)")
        ax1.set_ylabel("Silhouette")
        ax1.grid()

        # --- Calinski-Harabasz ---
        ax2.plot(ks, results["calinski"], marker='o', color='green')
        ax2.set_title("Calinski-Harabasz (Maior é melhor)")
        ax2.set_xlabel("K (Componentes)")
        ax2.set_ylabel("Score")
        ax2.grid()

        # --- BIC ---
        ax3.plot(ks, results["bic"], marker='o', color='purple')
        ax3.set_title("BIC (Menor é melhor)")
        ax3.set_xlabel("K (Componentes)")
        ax3.set_ylabel("BIC Score")
        ax3.grid()

        plt.tight_layout()
        plt.show()

    return {
        "best_k": best_k,
        "scores": results,
        "best_models": best_models
    }

In [ ]:
resultado_gmm = find_optimal_k_gmm(X_cluster, k_min=2, k_max=10)

In [ ]:
gmm = GaussianMixture(n_components=2, random_state=42, n_init=10)
gmm.fit(X_cluster)
labels_gmm = gmm.predict(X_cluster)

In [ ]:
X_umap = visualize_umap(X_cluster, labels=labels_gmm, n_neighbors=5, min_dist=0.1, n_components=2, metric='euclidean', random_state=42, verbose=False)

In [ ]:
X_tsne = visualize_tsne(X_cluster, labels=labels_gmm, perplexity=5, learning_rate='auto', max_iter=10000, random_state=42, method='exact', angle=0.5, metric='euclidean', verbose=0)

#### Plot variables by clusters

In [ ]:
plot_variable_by_cluster(
    df_data=df_dataset_anomalies, 
    labels=labels_gmm, 
    variable_name='torque_ds', # nome da variavel
    timestamp_col=timestamp # variável de timestamp
)

### Hierachical Clustering

In [ ]:
import scipy.cluster
import scipy.cluster.hierarchy as hierarchy

Z = hierarchy.linkage(X_cluster, method='ward')
hierarchy.dendrogram(Z, color_threshold=0.3);

In [ ]:
def find_optimal_k_hierarchical(X_scaled, k_min=2, k_max=10, plot=True):
    """
    Busca automática do K ótimo para Hierarchical Clustering (Agglomerative).
    
    Calcula:
    - Silhouette
    - Calinski-Harabasz
    - Davies-Bouldin
    - Pseudo-Inertia (WSS - Soma dos Quadrados Intra-Cluster)

    Args:
        k_min (int): Mínimo de clusters (>= 2)
        k_max (int): Máximo de clusters
    """

    ks = list(range(k_min, k_max + 1))

    # Armazena métricas
    results = {
        "silhouette": [],
        "calinski": [],
        "davies": [],
        "wss": [] # "Within-Cluster Sum of Squares" (Pseudo-Inertia)
    }

    # Para salvar o modelo final por métrica
    best_models = {}

    for k in ks:
        # linkage='ward' minimiza a variância dos clusters sendo fundidos
        hc = AgglomerativeClustering(n_clusters=k, linkage='ward')
        labels = hc.fit_predict(X_scaled)

        # Métricas de validação
        results["silhouette"].append(silhouette_score(X_scaled, labels))
        results["calinski"].append(calinski_harabasz_score(X_scaled, labels))
        results["davies"].append(davies_bouldin_score(X_scaled, labels))

        # --- Cálculo Manual da WSS (Para simular o Cotovelo) ---
        # Necessário pois AgglomerativeClustering não expõe .inertia_
        wss_val = 0
        for cluster_id in range(k):
            cluster_points = X_scaled[labels == cluster_id]
            if len(cluster_points) > 0:
                centroid = cluster_points.mean(axis=0)
                wss_val += np.sum((cluster_points - centroid) ** 2)
        results["wss"].append(wss_val)

    # -----------------------------
    # Determinação de cada K ótimo
    # -----------------------------
    best_k = {}

    best_k["silhouette"] = ks[np.argmax(results["silhouette"])]
    best_k["calinski"]   = ks[np.argmax(results["calinski"])]
    best_k["davies"]     = ks[np.argmin(results["davies"])]
    
    # Método do Cotovelo (simplificado na segunda derivada da WSS)
    diffs = np.diff(results["wss"])
    second_diffs = np.diff(diffs)
    elbow_idx = np.argmin(second_diffs) 
    best_k["elbow"] = ks[elbow_idx + 1] if (elbow_idx + 1) < len(ks) else ks[elbow_idx]

    # Modelos finais
    for metric_name, k_value in best_k.items():
        best_models[metric_name] = AgglomerativeClustering(n_clusters=k_value, linkage='ward').fit(X_scaled)

    # -----------------------------
    # Plot em subplots
    # -----------------------------
    if plot:
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        ax1, ax2, ax3, ax4 = axes.ravel()

        # --- Silhouette ---
        ax1.plot(ks, results["silhouette"], marker='o', color='blue')
        ax1.set_title("Silhouette Score (Maior é melhor)")
        ax1.set_xlabel("K")
        ax1.set_ylabel("Silhouette")
        ax1.grid()

        # --- Calinski-Harabasz ---
        ax2.plot(ks, results["calinski"], marker='o', color='green')
        ax2.set_title("Calinski-Harabasz (Maior é melhor)")
        ax2.set_xlabel("K")
        ax2.set_ylabel("Score")
        ax2.grid()

        # --- Davies-Bouldin ---
        ax3.plot(ks, results["davies"], marker='o', color='red')
        ax3.set_title("Davies-Bouldin (Menor é melhor)")
        ax3.set_xlabel("K")
        ax3.set_ylabel("Score")
        ax3.grid()

        # --- WSS (Elbow) ---
        ax4.plot(ks, results["wss"], marker='o', color='purple')
        ax4.set_title("Pseudo-Inertia / WSS (Método do Cotovelo)")
        ax4.set_xlabel("K")
        ax4.set_ylabel("Soma Quadrados Distância")
        ax4.grid()

        plt.tight_layout()
        plt.show()

    return {
        "best_k": best_k,
        "scores": results,
        "best_models": best_models
    }

In [ ]:
resultado_hc = find_optimal_k_hierarchical(X_cluster, k_min=2, k_max=10)

In [ ]:
hc = AgglomerativeClustering(n_clusters=2, linkage='ward')
labels_hierarchical = hc.fit_predict(X_cluster)

In [ ]:
X_umap = visualize_umap(X_cluster, labels=labels_hierarchical, n_neighbors=5, min_dist=0.1, n_components=2, metric='euclidean', random_state=42, verbose=False)

In [ ]:
X_tsne = visualize_tsne(X_cluster, labels=labels_hierarchical, perplexity=5, learning_rate='auto', max_iter=10000, random_state=42, method='exact', angle=0.5, metric='euclidean', verbose=0)

#### Plot variables by clusters

In [ ]:
plot_variable_by_cluster(
    df_data=df_dataset_anomalies, 
    labels=labels_hierarchical, 
    variable_name='torque_ds', # nome da variavel
    timestamp_col=timestamp # variável de timestamp
)

### BIRCH

In [ ]:
def find_optimal_k_birch(X_scaled, k_min=2, k_max=10, threshold=0.5, plot=True):
    """
    Busca automática do K ótimo para BIRCH.
    
    O BIRCH constrói uma árvore CF (Clustering Feature) e depois aplica
    um clusterizador hierárquico nos sub-clusters das folhas.

    Args:
        threshold (float): Raio máximo do sub-cluster. Se os dados forem normalizados,
                           0.5 é um bom ponto de partida.
    """

    ks = list(range(k_min, k_max + 1))

    # Armazena métricas
    results = {
        "silhouette": [],
        "calinski": [],
        "davies": [],
        "wss": [] 
    }

    # Para salvar o modelo final por métrica
    best_models = {}

    for k in ks:
        # BIRCH reduz os dados para sub-clusters e depois aplica AgglomerativeClustering globalmente para chegar em K
        birch = Birch(n_clusters=k, threshold=threshold)
        labels = birch.fit_predict(X_scaled)

        # Métricas de validação
        results["silhouette"].append(silhouette_score(X_scaled, labels))
        results["calinski"].append(calinski_harabasz_score(X_scaled, labels))
        results["davies"].append(davies_bouldin_score(X_scaled, labels))

        # --- Cálculo Manual da WSS (Pseudo-Inertia) ---
        # O BIRCH foca em sub-clusters, mas para avaliar o K final,
        # calculamos a compactação baseada nos labels finais gerados.
        wss_val = 0
        for cluster_id in range(k):
            cluster_points = X_scaled[labels == cluster_id]
            if len(cluster_points) > 0:
                centroid = cluster_points.mean(axis=0)
                wss_val += np.sum((cluster_points - centroid) ** 2)
        results["wss"].append(wss_val)

    # -----------------------------
    # Determinação de cada K ótimo
    # -----------------------------
    best_k = {}

    best_k["silhouette"] = ks[np.argmax(results["silhouette"])]
    best_k["calinski"]   = ks[np.argmax(results["calinski"])]
    best_k["davies"]     = ks[np.argmin(results["davies"])]
    
    # Método do Cotovelo
    diffs = np.diff(results["wss"])
    second_diffs = np.diff(diffs)
    elbow_idx = np.argmin(second_diffs) 
    best_k["elbow"] = ks[elbow_idx + 1] if (elbow_idx + 1) < len(ks) else ks[elbow_idx]

    # Modelos finais
    for metric_name, k_value in best_k.items():
        best_models[metric_name] = Birch(n_clusters=k_value, threshold=threshold).fit(X_scaled)

    # -----------------------------
    # Plot em subplots
    # -----------------------------
    if plot:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        ax1, ax2, ax3, ax4 = axes.ravel()

        # --- Silhouette ---
        ax1.plot(ks, results["silhouette"], marker='o', color='blue')
        ax1.set_title("Silhouette Score (Maior é melhor)")
        ax1.set_xlabel("K")
        ax1.set_ylabel("Silhouette")
        ax1.grid()

        # --- Calinski-Harabasz ---
        ax2.plot(ks, results["calinski"], marker='o', color='green')
        ax2.set_title("Calinski-Harabasz (Maior é melhor)")
        ax2.set_xlabel("K")
        ax2.set_ylabel("Score")
        ax2.grid()

        # --- Davies-Bouldin ---
        ax3.plot(ks, results["davies"], marker='o', color='red')
        ax3.set_title("Davies-Bouldin (Menor é melhor)")
        ax3.set_xlabel("K")
        ax3.set_ylabel("Score")
        ax3.grid()

        # --- WSS (Elbow) ---
        ax4.plot(ks, results["wss"], marker='o', color='purple')
        ax4.set_title("Pseudo-Inertia / WSS (Método do Cotovelo)")
        ax4.set_xlabel("K")
        ax4.set_ylabel("Soma Quadrados Distância")
        ax4.grid()

        plt.tight_layout()
        plt.show()

    return {
        "best_k": best_k,
        "scores": results,
        "best_models": best_models
    }

In [ ]:
# K=2 até K=10 com threshold padrão de 0.5
resultado_birch = find_optimal_k_birch(X_cluster)

In [ ]:
birch = Birch(n_clusters=2, threshold=0.5)
labels_birch = birch.fit_predict(X_cluster)

In [ ]:
X_umap = visualize_umap(X_cluster, labels=labels_birch, n_neighbors=5, min_dist=0.1, n_components=2, metric='euclidean', random_state=42, verbose=False)

In [ ]:
X_tsne = visualize_tsne(X_cluster, labels=labels_birch, perplexity=5, learning_rate='auto', max_iter=10000, random_state=42, method='exact', angle=0.5, metric='euclidean', verbose=0)

#### Plot variables by clusters

In [ ]:
plot_variable_by_cluster(
    df_data=df_dataset_anomalies, 
    labels=labels_birch, 
    variable_name='torque_ds', # nome da variavel
    timestamp_col=timestamp # variável de timestamp
)

## Mc2PCA

In [ ]:
from tqdm import tqdm
import pickle
from scipy.spatial.distance import cosine
from dtw import dtw

class Mc2PCA() :
    def __init__(self, 
                    K : int,
                    p : int,
                    epsilon : float = 1e-7,
                    max_iter : int = 100, 
                    distance_metric : str = 'euclidean') :
        """
        Perform the Mc2PCA algorithm on the given DataFrame or NumPy array.
        Implementation following the algorithm described in the paper:
        Li, H. (2019). Multivariate time series clustering based on common principal component analysis. Neurocomputing, 349.

        Args: 
            K (int): The number of clusters to form using k-means.
            p (int): The number of principal components to retain in CPCA.
            epsilon (float): The threshold for convergence. 
            max_iter (int, optional): The maximum number of iterations for the clustering algorithm. Defaults to 100.
            distance_metric (str, optional): The distance metric to use for the clustering algorithm. The values can be: 'euclidean', 'cosine', 'dtw', 'l1'. Defaults to 'euclidean'.
            S (list of ndarray): A list of K arrays, each array containing the common space of the kth cluster.
            idx (list of list of int): A list of K lists, each containing the indices of the samples in the kth cluster.
            E (list of float): A list containing the errors at each iteration.
            info_by_cluster (list of float): A list containing the information percentage retained by each cluster.
        """
        self.K = K
        self.p = p
        self.epsilon = epsilon
        self.max_iter = max_iter
        self.distance_metric = distance_metric
        self.S = None
        self.idx = None
        self.E = None
        self.info_by_cluster = None


    def fit(self, X : np.ndarray or pd.DataFrame):
        """
        Fit the model to the given data.

        Args:
            X (DataFrame or ndarray): The input MTS that can be stored as a DataFrame containing the data with samples as 
                                        rows and variables as columns, and each cell containing a pandas Series 
                                        object or a numpy ndarray, OR a 2D NumPy array with the same shape and containing
                                        a 1D NumPy array in each cell.
        """

        # if X is a dataframe, convert into npy array
        if isinstance(X, pd.DataFrame):
            X = convert_to_numpy(X)

        # Center the data
        X = center_data(X)
        # Compute the covariance matrices of each time series
        cov_matrices = compute_covariance_matrices(X)
    
        # Initialize the indices
        idx = np.array_split(np.arange(X.shape[0]), self.K)
        # Initialize the associated common spaces
        S, _ = compute_common_spaces(cov_matrices,idx,self.p)

        # Store the errors
        E = [np.inf]

        for t in tqdm(range(1, self.max_iter + 1), leave=False):

            # Assign the clusters based on k-means
            I,v = assign_clusters(X,S,self.K, distance_metric= self.distance_metric)
            E.append(np.sum(v)/len(v)) # normalize the error

            # Check convergence
            if np.abs(E[t-1] - E[t]) < self.epsilon:
                break
            
            # Assign new clusters
            idx = [np.where(I == k)[0] for k in range(self.K)]

            # Compute the new common spaces after the assignment
            S, info_by_cluster = compute_common_spaces(cov_matrices,idx,self.p)

        # Store the results in the class attributes
        self.info_by_cluster = info_by_cluster
        self.idx = idx
        self.E = E
        self.S = S
        return idx, E, info_by_cluster

    def inference(self, X_test : np.ndarray or pd.DataFrame):
        """  
        Perform inference on the given test set using the learned model.

        Args:
            X_test (DataFrame or ndarray): The input MTS that can be stored as a DataFrame containing the data with samples as 
                        rows and variables as columns, and each cell containing a pandas Series 
                        object or a numpy ndarray, OR a 2D NumPy array with the same shape and containing
                        a 1D NumPy array in each cell.
            
        """

        # if X is a dataframe, convert into npy array
        if isinstance(X_test, pd.DataFrame):
            X_test = convert_to_numpy(X_test)  

        # Center the data
        X_test = center_data(X_test)

        # Assign the clusters based on k-means using the learned common spaces
        I, _ = assign_clusters(X_test, self.S, self.K, distance_metric = self.distance_metric)
        
        # Assign new clusters
        idx = [np.where(I == k)[0] for k in range(self.K)]

        return idx
    
    def save_model(self, path: str):
        """
        Save the model using pickle.

        Args:
            path (str): The path to the file where the model should be saved.
        """
        with open(path, 'wb') as file:
            pickle.dump(self, file)
        

    def load_model(cls, path: str):
        """
        Load a model using pickle.

        Args:
            path (str): The path to the file from which the model should be loaded.

        Returns:
            Mc2PCA: The loaded model.
        """
        with open(path + '.pkl', 'rb') as file:
            return pickle.load(file)


def convert_to_numpy(df):
    """
    Convert a DataFrame where each cell contains a pandas Series into a 1D NumPy array.

    Args:
        df (DataFrame): The input DataFrame containing the data with samples as rows and variables as columns, 
                        and each cell containing a pandas Series object, or a numpy ndarray.
    
    Returns:
        ndarray: The 2D NumPy array containing the data with samples as rows and variables as columns, 
                 and each cell containing a 1D NumPy array.
    """

    # First convert the pandas series to ndarray if necessary
    if isinstance(df.iloc[0,0], pd.Series):
        for col_name in df.columns:
            df[col_name] = df[col_name].apply(lambda series: series.to_numpy() if series is not None else np.nan)

    df_npy = df.to_numpy()

    return df_npy
 

def center_data(X) :
    """
    Center the data by subtracting the mean of each time series from each cell.

    Args:
        X (ndarray): The input 2D NumPy array containing the multivariate time series data with samples as rows and variables as columns, and each cell containing a 1D numpy ndarray.
    
    Returns:
        ndarray: The centered 2D NumPy array.
    """
    n, m = X.shape  # Number of samples and features
    centered_X = np.empty_like(X)

    for i in range(n):
        for j in range(m):
            time_series = X[i, j]
            mean = np.mean(time_series)
            centered_X[i, j] = time_series - mean

    return centered_X
         

def compute_covariance_matrices(centered_X):
    """
    Compute the covariance matrix of each time series in the given 2D Numpy array.

    Args:
        centered_X (ndarray): The input 2D NumPy array containing the centered multivariate time series data, 
                              where each cell contains a 1D NumPy array representing a time series.
    
    Returns:
        list: A list containing the covariance matrix of each time series in the given array.
    """
    n,m = centered_X.shape

    # Store the covariance matrix of each time series
    cov_matrices = []

    for i in range(n):
        # Extract and stack each time series in the row into a 2D array
        row_data = np.column_stack([centered_X[i, j] for j in range(m) if centered_X[i, j] is not None])

        # Compute the covariance matrix of the time seriif row_data.size > 0:
        cov_matrix = np.cov(row_data.T, bias=True)  
       
        cov_matrices.append(cov_matrix)

    return cov_matrices



def CPCA(Sigma, p):
    """
    Perform Common Principal Component Analysis on a set of covariance matrices corresponding to
    a cluster of a multivariate time series, and return the common space of the cluster.

    Args:
        Sigma (list of ndarray): The list of covariance matrices.
        p (int): The number of principal components to retain.

    Returns:
        ndarray: The common space of the cluster.
    """
    mean_cov = np.mean(Sigma, axis=0) # mean covariance matrix of the cluster
    _, val_propre, vt = np.linalg.svd(mean_cov) # SVD of the mean covariance matrix
    val_propre = val_propre**2 # eigenvalues of the mean covariance matrix
    prct_info =  np.sum(val_propre[:p]/np.sum(val_propre))
    return vt[:p,:].T, prct_info # return the first p principal components and the information retained


def compute_common_spaces(cov_matrices,cluster_indices,p):
    """
    Compute the common principal components of each cluster using CPCA.

    This function iterates over the provided cluster indices and applies CPCA to the 
    covariance matrices corresponding to each cluster. This is used to find a common 
    subspace for each cluster that captures the most variance.

    Args:
        cov_matrices (list of ndarray): A list of covariance matrices for all samples in the dataset.
        cluster_indices (list of list of int): A list of K lists, each containing the indices of the samples in the kth cluster.
        p (int): The number of principal components to retain in CPCA.

    Returns:
        list of ndarray: A list of K arrays, each array containing the common space of the kth cluster.
    """
    S = []
    info_by_cluster = []
    for indices in cluster_indices:
        if len(indices) > 0: # ensure that the cluster is not empty
            vec_propres, prct_info = CPCA([cov_matrices[i] for i in indices], p)
            S.append(vec_propres)
            info_by_cluster.append(prct_info)
        else:
            S.append(None)
    return S, info_by_cluster


def assign_clusters(X,S,K, distance_metric='euclidean'):
    """
    Assign each multivariate time series to a cluster based on the reconstruction error.

    Compute the reconstruction error for each time series after projecting it onto the common space of each cluster.
    Each time series is then assigned to the cluster for which it has the lowest reconstruction error.
    For empty clusters (very unlikely), assign a high error value.

    Args:
        X (ndarray): The input array containing the centered multivariate time series as (nb_samples,nb_variables) 
                        where each cell contains a 1D NumPy array representing a time series.
        S (list of ndarray): A list of K arrays, each array containing the common space of the kth cluster.
        K (int): The number of clusters.
        distance_metric (str, optional): The distance metric to use for error in the clustering algorithm. The values can be: 'euclidean', 'cosine', 'dtw', 'l1'. Defaults to 'euclidean'.
    
    Returns:
        tuple: A tuple containing two elements:
            - ndarray: An array containing the indices of the clusters to which each time series is assigned.
            - ndarray: An array containing containing the minimum reconstruction error for each time series.
    """
    n = X.shape[0]
    Error = np.zeros((n, K))
    
    for k in range(K):
        if S[k] is not None:
            sst = np.matmul(S[k], S[k].T)
            for i in range(n):
                time_series = np.column_stack(X[i, :])  # Stacking the 1D arrays in the row into a 2D array (length,nb_variables)
                Y = np.matmul(time_series, sst)
                if distance_metric == 'euclidean':
                    err = np.linalg.norm(time_series - Y, axis=1)
                elif distance_metric == 'cosine':
                    err = np.array([cosine(time_series[j], Y[j]) for j in range(time_series.shape[0])])
                elif distance_metric == 'dtw':
                    err = np.array([dtw(time_series[j],Y[j]).distance for j in range(time_series.shape[0])])
                elif distance_metric == 'l1':
                    err = np.linalg.norm(time_series - Y, ord=1, axis=1)
                Error[i, k] = np.mean(err)  # Mean error for the time series
        else:
            Error[:, k] = np.inf

    I = np.argmin(Error, axis=1)
    v = Error[np.arange(n), I]
    return I, v

In [ ]:
def prepare_raw_df_for_mc2pca(df_raw, anomaly_periods, timestamp_col=None):
    """
    Transforma um DataFrame bruto (N_linhas, M_vars) em uma matriz aninhada (N_anomalias, M_vars)
    compatível com Mc2PCA, aplicando normalização global.

    Args:
        df_raw (pd.DataFrame): DataFrame com valores brutos das variáveis.
        anomaly_periods (list): Lista de tuplas [(inicio, fim), ...].
        timestamp_col (str, opcional): Nome da coluna de tempo. Se None, assume que o índice é o tempo.

    Returns:
        np.ndarray: Array de objetos (N_anomalias, M_variaveis) onde cada célula é um array 1D.
    """
    
    df_work = df_raw.copy()
    
    if timestamp_col:
        if timestamp_col in df_work.columns:
            df_work.set_index(timestamp_col, inplace=True)
            df_work.sort_index(inplace=True)

    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(df_work)
    df_scaled = pd.DataFrame(data_scaled, columns=df_work.columns, index=df_work.index)

    N_samples = len(anomaly_periods)
    M_features = df_scaled.shape[1]
    
    X_nested = np.empty((N_samples, M_features), dtype=object)
    
    valid_count = 0
    for i, (start, end) in enumerate(anomaly_periods):
        try:
            subset = df_scaled.loc[start:end]
            
            if subset.empty:
                print(f"  [Aviso] Anomalia {i} ({start} - {end}) vazia ou não encontrada no índice.")
                for j in range(M_features):
                    X_nested[i, j] = np.zeros(1) 
                continue

            for j in range(M_features):
                X_nested[i, j] = subset.iloc[:, j].values
            
            valid_count += 1
            
        except Exception as e:
            print(f"  [Erro] Falha ao processar anomalia {i}: {e}")

    return X_nested

df_dataset_anomalies_ids = df_dataset_anomalies.copy()
df_dataset_anomalies.drop(columns="anomaly_id", inplace=True, errors="ignore")
X_train_mc2pca = prepare_raw_df_for_mc2pca(
    df_raw=df_dataset_anomalies, 
    anomaly_periods=anomaly_periods, 
    timestamp_col=timestamp
)
print(f"Shape: {X_train_mc2pca.shape}")

In [ ]:
K = 2
p = 5
seuil = 1e-7
model = Mc2PCA(K, p, seuil, max_iter=1000, distance_metric="euclidean") ## euclidean | dtw
clusters, E, info_by_cluster = model.fit(X_train_mc2pca)

In [ ]:
## explained variance per cluster
print(info_by_cluster)

In [ ]:
## resconstruction error
plt.plot(E[1:]) 
plt.title("Curva de Convergência (Erro de Reconstrução)")
plt.xlabel("Iteração")
plt.ylabel("Erro Médio (DTW)")
plt.show()

In [ ]:
## format clusters
n_samples = X_train_mc2pca.shape[0]
labels_mc2pca = np.zeros(n_samples, dtype=int)

for cluster_id, indices in enumerate(clusters):
    labels_mc2pca[indices] = cluster_id

#### Plot variables by clusters

In [ ]:
plot_variable_by_cluster(
    df_data=df_dataset_anomalies_ids, 
    labels=labels_mc2pca, 
    variable_name='torque_ds', # nome da variavel
    timestamp_col=timestamp # variável de timestamp
)

## Raw TimeSeries Based (Custo computacional alto)

In [ ]:
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.utils import to_time_series_dataset
from tslearn.clustering import TimeSeriesKMeans

df_subset = df_dataset.copy()
df_subset.set_index(timestamp, inplace=True)

lista_anomalias = []
for (ini, fim) in anomaly_periods:
    df_period = df_subset.loc[ini:fim]
    
    # ignora anomalias muito curtas (ex: < 5 pontos)
    if len(df_period) > 5:
        # Adiciona apenas os valores (numpy array) à lista
        lista_anomalias.append(df_period.values)

X_formatted = to_time_series_dataset(lista_anomalias)

scaler = TimeSeriesScalerMeanVariance(mu=0., std=1.)
X_scaled_tslearn = scaler.fit_transform(X_formatted) ## (nº anomalias, nº de amostras, nº de variáveis)
print(f"Shape dos dados: {X_scaled_tslearn.shape}")

In [ ]:
X_scaled_aeon = X_scaled_tslearn.transpose(0, 2, 1)
print(f"Shape dos dados: {X_scaled_aeon.shape}")

### TimeSeries KMeans + DTW

In [ ]:
inertias = []
ks = range(1, 5) # test number os clusters

print("Calculando Inércia para diferentes k...")

for k in ks:
    # Use os mesmos parâmetros do seu modelo final
    model = TimeSeriesKMeans(n_clusters=k, metric="softdtw", max_iter_barycenter=5, random_state=42, n_jobs=-1)
    model.fit(X_scaled_tslearn)
    inertias.append(model.inertia_)

# Plot
plt.figure(figsize=(8, 4))
plt.plot(ks, inertias, 'bo-')
plt.xlabel('Número de Clusters (k)')
plt.ylabel('Inércia (DTW)')
plt.title('Método do Cotovelo')
plt.grid(True)
plt.show()

In [ ]:
n_clusters = 2

dba_km = TimeSeriesKMeans(n_clusters=n_clusters,
                          n_init=2,
                          metric="softdtw",
                          verbose=False,
                          max_iter_barycenter=10,
                          random_state=42,
                          n_jobs=-1)

labels_kmeansDTW = dba_km.fit_predict(X_scaled_tslearn)

In [ ]:
labels_kmeansDTW

#### Plot variables by clusters

In [ ]:
plot_variable_by_cluster(
    df_data=df_dataset_anomalies_ids, 
    labels=labels_kmeansDTW, 
    variable_name='torque_ds', # nome da variavel
    timestamp_col=timestamp # variável de timestamp
)

### TimeSeriesKMedoids

In [ ]:
## cant handle with mising values and uneqal timeseries lenght
from aeon.clustering import TimeSeriesKMedoids

kmedoids = TimeSeriesKMedoids(n_clusters=3, distance="msm", random_state=42)
resultado_kmedoids = kmedoids.fit_predict(X_scaled_aeon)
resultado_kmedoids

### KShape

In [ ]:
from kshape.core import KShapeClusteringCPU
num_clusters = 3

ksc = KShapeClusteringCPU(num_clusters, centroid_init='zero', max_iter=100, n_jobs=-1)
ksc.fit(X_formatted)

labels = ksc.labels_
centroids = ksc.centroids_